<a href="https://colab.research.google.com/github/ozair247/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ozair247/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

**Lane 2 — Refresh / Content Opportunity Scoring.** This is the baseline my Week-5 model must beat: two signal checks, one transparent rule (score + one reason code + one action label), a ranked queue, and a hand review of my own top 10.

Setup used throughout: DuckDB reading remote Parquet directly (see `skills/querying-big-datasets`), iterating on `month=2026-03` (mid-panel), same as `w03_data_contract.ipynb`. `month=2026-06` and the `_sample` table stay sealed — not touched anywhere in this notebook.

## 0. Setup — connect to the warehouse

Same connection pattern as `w03_data_contract.ipynb`.

In [21]:
%pip install -q duckdb

import duckdb
import pandas as pd
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

FACT_MONTH = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"
DIM_CONTENT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')"
DIM_CLIENTS = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet')"

print("DuckDB connected. Confirming the March partition is visible...")
print(con.sql(f"SELECT COUNT(*) AS rows, MIN(report_date) AS min_d, MAX(report_date) AS max_d FROM {FACT_MONTH}").df())


DuckDB connected. Confirming the March partition is visible...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

      rows      min_d      max_d
0  9841378 2026-03-01 2026-03-31


### If a query below errors on an unknown column

My sandbox has no network path to `huggingface.co`, so — same caveat as `w03` — I built this against the column names confirmed in `skills/flyrank/flyrank-data/SKILL.md` and against `w03`'s already-verified query shapes, not a live schema check. Run the probe below once if anything 404s or errors on a column name, then fix the name in every cell that uses it.

In [22]:
# Schema probe — run once if a later query errors on an unknown column
print("dim_content columns:")
print(con.sql(f"DESCRIBE SELECT * FROM {DIM_CONTENT} LIMIT 1").df())


dim_content columns:
                   column_name column_type null   key default extra
0               client_hash_id     VARCHAR  YES  None    None  None
1              content_hash_id     VARCHAR  YES  None    None  None
2              keyword_hash_id     VARCHAR  YES  None    None  None
3                  url_hash_id     VARCHAR  YES  None    None  None
4           keyword_char_count      BIGINT  YES  None    None  None
5          keyword_token_count      BIGINT  YES  None    None  None
6               url_char_count      BIGINT  YES  None    None  None
7         content_created_date        DATE  YES  None    None  None
8         content_updated_date        DATE  YES  None    None  None
9                 content_type     VARCHAR  YES  None    None  None
10               search_volume      BIGINT  YES  None    None  None
11                 competition      DOUBLE  YES  None    None  None
12           competition_level     VARCHAR  YES  None    None  None
13                         

## 1. Two signal checks, then my rule

### Build the March feature frame + proxy label

Same shape as `w03_data_contract.ipynb`: one row per `content_hash_id x client_hash_id`, March 2026, `is_declining_proxy` from last-15-day vs. prior-15-day impressions inside the month. Nothing here — `content_age_days`, `impressions_month`, `ctr_month`, `avg_position_month` — is derived from the last15/prev15 split that makes the label, so none of it is label-derived.

In [23]:
feature_frame = con.sql(f"""
    SELECT
        f.content_hash_id,
        f.client_hash_id,
        SUM(f.gsc_impressions)                                      AS impressions_month,
        SUM(f.gsc_clicks)                                           AS clicks_month,
        AVG(NULLIF(f.gsc_avg_position, 0))                          AS avg_position_month,
        SUM(f.gsc_clicks) / NULLIF(SUM(f.gsc_impressions), 0) * 100 AS ctr_month,
        DATE_DIFF('day', ANY_VALUE(c.content_created_date), DATE '2026-03-31') AS content_age_days,
        SUM(CASE WHEN f.report_date >= DATE '2026-03-16' THEN f.gsc_impressions ELSE 0 END) AS impr_last15,
        SUM(CASE WHEN f.report_date <  DATE '2026-03-16' THEN f.gsc_impressions ELSE 0 END) AS impr_prev15
    FROM {FACT_MONTH} f
    LEFT JOIN {DIM_CONTENT} c ON f.content_hash_id = c.content_hash_id
    GROUP BY f.content_hash_id, f.client_hash_id
    HAVING SUM(f.gsc_impressions) > 0
""").df()

feature_frame["is_declining_proxy"] = (
    (feature_frame["impr_prev15"] > 0)
    & ((feature_frame["impr_last15"] - feature_frame["impr_prev15"]) / feature_frame["impr_prev15"] < -0.20)
).astype(int)

print(f"Rows: {len(feature_frame)}")
print(f"Base rate (is_declining_proxy): {feature_frame['is_declining_proxy'].mean():.1%}")
feature_frame.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 176738
Base rate (is_declining_proxy): 28.1%


,content_hash_id,client_hash_id,impressions_month,clicks_month,avg_position_month,ctr_month,content_age_days,impr_last15,impr_prev15,is_declining_proxy
0,content_d0dff76c889de68f,client_62f4a7e64f5e0096,181.0,0.0,5.331238,0.000000,47,70.0,111.0,1
1,content_ac8663da7484669a,client_62f4a7e64f5e0096,34.0,0.0,6.419872,0.000000,47,14.0,20.0,1
2,content_39d7361b4945d504,client_62f4a7e64f5e0096,77.0,0.0,4.888929,0.000000,47,20.0,57.0,1
3,content_d49a012dcb924e31,client_62f4a7e64f5e0096,329.0,0.0,5.177774,0.000000,47,83.0,246.0,1
4,content_cec711b02f3bbde6,client_62f4a7e64f5e0096,602.0,4.0,4.428747,0.664452,47,403.0,199.0,0


### Signal check 1 — staleness (behind the `stale_visible_page` refresh flag)

**Note on the substitution:** the warehouse's `dim_content` ships `content_created_date`, not a last-updated date, so I use `content_age_days` as my staleness proxy — directional for the product's `days_since_last_update`, not identical to it. Same substitution `w03` already made and Ozair ran without error.

**Claim:** older content is more likely to be flagged `is_declining_proxy` — the reasoning behind `stale_visible_page` (`days_since_last_update >= 180`).

**Test:** bucket `content_age_days` at the product's own cutoff (180 days) plus one finer split, and read the proxy-decline rate and `n` in each bucket.

In [24]:
import numpy as np

age_bins = [0, 90, 180, 365, 730, np.inf]
age_labels = ["<90", "90-180", "180-365", "365-730", "730+"]
feature_frame["age_bucket"] = pd.cut(feature_frame["content_age_days"], bins=age_bins, labels=age_labels)

age_table = feature_frame.groupby("age_bucket", observed=True).agg(
    n=("is_declining_proxy", "size"),
    decline_rate=("is_declining_proxy", "mean"),
).reset_index()
print(age_table)
print()
print(f"Base rate for reference: {feature_frame['is_declining_proxy'].mean():.1%}")


  age_bucket      n  decline_rate
0        <90  57705      0.243151
1     90-180  26247      0.375395
2    180-365  71046      0.297385
3    365-730  21710      0.214694

Base rate for reference: 28.1%


Verdict: MIXED

The decline rate does not rise monotonically with age.  <90 days shows 24.3%, 90–180 days jumps to 37.5%, but then drops to 29.7% for 180–365 days and 21.5% for 365–730 days.  
Staleness helps up to ~180 days but older content actually declines less — perhaps very old, surviving pages have stabilised. The 730+ bucket is missing (zero rows), so no data there.  
Thus age alone is a weak signal; a rule must combine it with another factor.

### Signal check 2 — CTR (behind the `low_ctr_visible_page` / CTR-fix flag)

**Claim:** pages with weak CTR for their position are more likely `is_declining_proxy` — the reasoning behind `low_ctr_visible_page` (`impressions_90d >= 500`, `0 < avg_position <= 20`, `ctr < 0.5`).

**Test:** restrict to the same visible, page-one/page-two slice the flag targets (`impressions_month >= 500`, `0 < avg_position_month <= 20`), bucket `ctr_month`, read decline rate and `n`.

In [25]:
ctr_slice = feature_frame[
    (feature_frame["impressions_month"] >= 500)
    & (feature_frame["avg_position_month"] > 0)
    & (feature_frame["avg_position_month"] <= 20)
].copy()

ctr_bins = [-np.inf, 0.5, 1.5, np.inf]
ctr_labels = ["low (<0.5)", "mid (0.5-1.5)", "high (1.5+)"]
ctr_slice["ctr_bucket"] = pd.cut(ctr_slice["ctr_month"], bins=ctr_bins, labels=ctr_labels)

ctr_table = ctr_slice.groupby("ctr_bucket", observed=True).agg(
    n=("is_declining_proxy", "size"),
    decline_rate=("is_declining_proxy", "mean"),
).reset_index()
print(f"Slice size (visible, position 1-20): {len(ctr_slice)} of {len(feature_frame)} rows")
print(ctr_table)


Slice size (visible, position 1-20): 50717 of 176738 rows
      ctr_bucket      n  decline_rate
0     low (<0.5)  41131      0.230021
1  mid (0.5-1.5)   8755      0.130668
2    high (1.5+)    831      0.090253


Verdict: FALSE

For visible pages (impressions ≥ 500, position 1–20), the decline rate is highest for the low CTR bucket (23.0%), then drops to 13.1% for mid CTR and 9.0% for high CTR.  
This is the **opposite** of what the original flag expects — low CTR is associated with *less* decline risk, not more. Therefore I should **not** use CTR as a signal for my refresh rule; it would push the queue in the wrong direction.

### My rule, in plain words
A page is worth reviewing for refresh if it is **both** old (content created ≥ 180 days ago) **and** still pulling meaningful search volume (≥ 500 impressions in March 2026).  
Old pages with no traffic are invisible and not worth a review slot; young pages with a traffic dip are not a staleness story.  
The score is simply `impressions_month` — higher traffic stale pages get higher priority.

## 2. Build the ranked queue (writes the CSV)

Transparent score, no fitted weights — same shape as the skill's worked example: `score = stale * visible * impressions`.

In [26]:
import os

df = feature_frame.copy()

stale   = (df["content_age_days"] >= 180).astype(int)
visible = (df["impressions_month"] >= 500).astype(int)

df["score"]       = stale * visible * df["impressions_month"]
df["reason_code"]  = np.where(df["score"] > 0, "stale_but_visible", "none")
df["action"]       = np.where(df["score"] > 0, "review_for_refresh", "no_action")

ranked = df.sort_values("score", ascending=False).reset_index(drop=True)

os.makedirs("work/outputs", exist_ok=True)
out_cols = ["content_hash_id", "client_hash_id", "score", "reason_code", "action",
            "content_age_days", "impressions_month", "ctr_month", "avg_position_month", "is_declining_proxy"]
ranked[out_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"Wrote {len(ranked)} rows to work/outputs/baseline_action_score.csv")
print(ranked["action"].value_counts())


Wrote 176738 rows to work/outputs/baseline_action_score.csv
action
no_action             145274
review_for_refresh     31464
Name: count, dtype: int64


In [27]:
# Precision@50, next to the base rate — the number Week-5's model has to beat
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

base_rate = ranked["is_declining_proxy"].mean()
p50 = precision_at_k(ranked["score"].values, ranked["is_declining_proxy"].values, 50)

print(f"Base rate: {base_rate:.3f}")
print(f"Baseline Precision@50: {p50:.3f}")
print(f"Lift over base rate: {p50 - base_rate:+.3f}")

import json
metrics = {
    "dev_month": "2026-03",
    "n_rows": int(len(ranked)),
    "base_rate": float(base_rate),
    "precision_at_50": float(p50),
    "rule": "score = (content_age_days>=180) * (impressions_month>=500) * impressions_month",
    "reason_code": "stale_but_visible",
    "signal_1_staleness_verdict": "MIXED",
    "signal_2_ctr_verdict": "FALSE",
}
with open("work/outputs/w04_baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("Wrote work/outputs/w04_baseline_metrics.json")


Base rate: 0.281
Baseline Precision@50: 0.140
Lift over base rate: -0.141
Wrote work/outputs/w04_baseline_metrics.json


## 3. Top-10 review

Read my own top ten with a skeptic's eye, one line each: the action, why it's there, what would make it wrong.

In [28]:
top10 = ranked.head(10)[["content_hash_id", "score", "reason_code", "action",
                          "content_age_days", "impressions_month", "ctr_month",
                          "avg_position_month", "is_declining_proxy"]]
top10


,content_hash_id,score,reason_code,action,content_age_days,impressions_month,ctr_month,avg_position_month,is_declining_proxy
0,content_eadb33b5df496f4a,617124.0,stale_but_visible,review_for_refresh,375,617124.0,0.918454,2.383011,0
1,content_ec2e0346994fb5a5,245276.0,stale_but_visible,review_for_refresh,434,245276.0,0.603402,2.854514,0
2,content_e8a52cf3d5988c07,244931.0,stale_but_visible,review_for_refresh,230,244931.0,0.273138,15.008339,1
3,content_0e03de7680314cd5,221310.0,stale_but_visible,review_for_refresh,375,221310.0,0.325336,2.675217,0
4,content_e7b5dd4dff461ad2,205045.0,stale_but_visible,review_for_refresh,343,205045.0,1.192909,4.544203,0
5,content_8d7d99f109e19aa2,203497.0,stale_but_visible,review_for_refresh,375,203497.0,0.142017,2.563756,0
6,content_36e53e9c707674fc,194579.0,stale_but_visible,review_for_refresh,229,194579.0,0.124371,32.766674,1
7,content_4ffe18112a5642e3,186983.0,stale_but_visible,review_for_refresh,375,186983.0,0.313397,2.331060,0
8,content_471d9cabce329a66,164885.0,stale_but_visible,review_for_refresh,375,164885.0,0.240167,4.656030,0
9,content_512dbad65bd5ade9,154358.0,stale_but_visible,review_for_refresh,187,154358.0,1.623499,3.019798,0


In [29]:
# Generate top-10 review text with real numbers
for i, (idx, row) in enumerate(top10.iterrows(), 1):
    age = row['content_age_days']
    impr = row['impressions_month']
    # Simple heuristic for "what would make it wrong"
    if age < 200:
        wrong = "Barely past the age cutoff; a small date correction could remove it."
    elif impr < 600:
        wrong = "Just above the visibility floor; seasonal dip could drop it next month."
    elif impr > 15000:
        wrong = "Could be a brand-driven page where staleness is irrelevant."
    else:
        wrong = "May be an evergreen article that naturally holds traffic without updates."

    print(f"Row {i} — action: review_for_refresh. Why: {age:.0f} days old, {impr:,.0f} impressions. "
          f"What would make it wrong: {wrong}")

Row 1 — action: review_for_refresh. Why: 375 days old, 617,124 impressions. What would make it wrong: Could be a brand-driven page where staleness is irrelevant.
Row 2 — action: review_for_refresh. Why: 434 days old, 245,276 impressions. What would make it wrong: Could be a brand-driven page where staleness is irrelevant.
Row 3 — action: review_for_refresh. Why: 230 days old, 244,931 impressions. What would make it wrong: Could be a brand-driven page where staleness is irrelevant.
Row 4 — action: review_for_refresh. Why: 375 days old, 221,310 impressions. What would make it wrong: Could be a brand-driven page where staleness is irrelevant.
Row 5 — action: review_for_refresh. Why: 343 days old, 205,045 impressions. What would make it wrong: Could be a brand-driven page where staleness is irrelevant.
Row 6 — action: review_for_refresh. Why: 375 days old, 203,497 impressions. What would make it wrong: Could be a brand-driven page where staleness is irrelevant.
Row 7 — action: review_for_r

**Row-by-row (using the real top‑10 values from the March 2026 queue):**

1. Row 1 — action: `review_for_refresh`. Why: 375 days old, 617,124 impressions — extremely high traffic, moderately stale. What would make it wrong: Could be a brand‑driven page (e.g., homepage, product category) where staleness is irrelevant; traffic may be navigational, not content‑driven.
2. Row 2 — action: `review_for_refresh`. Why: 434 days old, 245,276 impressions — very high visibility, well past the age threshold. What would make it wrong: Might be a hub page that gets refreshed by adding new child articles; the parent itself may not need updating.
3. Row 3 — action: `review_for_refresh`. Why: 230 days old, 244,931 impressions — just 50 days past the age cutoff but enormous traffic. What would make it wrong: If the page is a trending/timely piece that naturally decays, a refresh may not help.
4. Row 4 — action: `review_for_refresh`. Why: 375 days old, 221,310 impressions — similar to Row 1, high traffic and stale. What would make it wrong: Could be an evergreen guide that still ranks well without changes; “old” doesn’t mean “outdated.”
5. Row 5 — action: `review_for_refresh`. Why: 343 days old, 205,045 impressions — solid volume, over a year old. What would make it wrong: The decline signal might be seasonal; traffic could rebound without intervention.
6. Row 6 — action: `review_for_refresh`. Why: 375 days old, 203,497 impressions — another high‑traffic stale candidate. What would make it wrong: The page may have been silently updated (e.g., template change) without the `content_created_date` reflecting it.
7. Row 7 — action: `review_for_refresh`. Why: 229 days old, 194,579 impressions — borderline age but very visible. What would make it wrong: Only 49 days past the cutoff; a small data correction could push it under 180 days.
8. Row 8 — action: `review_for_refresh`. Why: 375 days old, 186,983 impressions — consistently stale and visible. What would make it wrong: If impressions are driven by a few high‑volume queries that are inherently stable, refresh won’t move the needle.
9. Row 9 — action: `review_for_refresh`. Why: 375 days old, 164,885 impressions — above the visibility floor, clearly stale. What would make it wrong: Could be a product page where freshness matters less than price/availability updates.
10. Row 10 — action: `review_for_refresh`. Why: 187 days old, 154,358 impressions — only 7 days past the age threshold, but still very visible. What would make it wrong: **Barely qualifies** – a one‑week shift in the creation date or a small drop in impressions next month would remove it from the queue entirely. Likely a false positive.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [30]:
# Identify the weakest pick (closest to thresholds among the top 10)
top10_stale = top10[top10['action'] == 'review_for_refresh'].copy()
top10_stale['age_margin'] = top10_stale['content_age_days'] - 180
top10_stale['impr_margin'] = top10_stale['impressions_month'] - 500
weakest = top10_stale.nsmallest(1, 'age_margin').iloc[0]  # closest to age cutoff
# Alternatively, choose by impression margin
# weakest = top10_stale.nsmallest(1, 'impr_margin').iloc[0]

print(f"**Weak pick:** Row with content_age_days = {weakest['content_age_days']:.0f}, "
      f"impressions_month = {weakest['impressions_month']:.0f}. "
      f"This is only {weakest['age_margin']:.0f} days past the 180‑day cutoff and has "
      f"{'modest' if weakest['impressions_month'] < 1000 else 'moderate'} volume. "
      f"Any small drop in impressions next month could remove it from the queue entirely; "
      f"likely a false positive.")

**Weak pick:** Row with content_age_days = 187, impressions_month = 154358. This is only 7 days past the 180‑day cutoff and has moderate volume. Any small drop in impressions next month could remove it from the queue entirely; likely a false positive.


In [31]:
# Leakage checklist, run out loud
future_window_cols = {"impr_last15", "impr_prev15"}
used_in_score = {"content_age_days", "impressions_month"}
product_flag_cols = set()  # health_score / priority_score / action_type were never queried above

print("Score inputs:", used_in_score)
print("Overlap with label-window columns (should be empty):", used_in_score & future_window_cols)
print("Overlap with product decision flags (should be empty):", used_in_score & product_flag_cols)
print()
print(f"Rows where the queue's own visible-impressions floor is barely cleared (500-600): "
      f"{((ranked['impressions_month'] >= 500) & (ranked['impressions_month'] < 600) & (ranked['action']=='review_for_refresh')).sum()}")


Score inputs: {'content_age_days', 'impressions_month'}
Overlap with label-window columns (should be empty): set()
Overlap with product decision flags (should be empty): set()

Rows where the queue's own visible-impressions floor is barely cleared (500-600): 2382


**Confirmed:** the score uses `content_age_days` and `impressions_month` only — neither is built from `impr_last15`/`impr_prev15` (the label's own ingredients) and no FlyRank product flag (`health_score`, `priority_score`, `action_type`) was queried anywhere above.

**Weak pick:** Row 10 is the weakest pick in the top 10. It has `content_age_days = 187` (only 7 days past the 180‑day cutoff) and `impressions_month = 154,358`. Although traffic is high, the staleness signal is extremely marginal. If the page was actually published a week later, or if next month’s impressions dip by 5–10% (which is normal fluctuation), it would no longer meet both thresholds and fall out of the queue. This is a classic false positive – the rule prioritises volume, but the staleness evidence is thin. A human reviewer would likely deprioritise it.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Both signal checks have a visible bucket table with `n`, and at least one is flag-linked (staleness / CTR-vs-position / volume)
- [ ] Both verdicts are one of CONFIRMED / OPPOSITE / MIXED / FALSE, with a plain-English reason
- [ ] The rule is ONE rule: a score, ONE reason code, an action label
- [ ] `work/outputs/baseline_action_score.csv` is written from the notebook (not committed — it's in `.gitignore`)
- [ ] `work/outputs/w04_baseline_metrics.json` is written and its verdict fields are filled in, not `FILL_IN`
- [ ] All ten top-10 rows are reviewed, each with a real "what would make it wrong"
- [ ] Weak picks section names an actual weak pick, not a hedge
- [ ] No future-window (`impr_last15`/`impr_prev15`) or product-flag inputs anywhere in the score
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all) — **run this in Colab**, my sandbox can't reach Hugging Face
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.